# LeetCode 867 — Transpose Matrix bằng Python

**Link đề:** https://leetcode.com/problems/transpose-matrix/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix, Simulation

```python
class Solution:
    def transpose(self, matrix: list[list[int]]) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Hiểu **chuyển vị (transpose)**: hàng thành cột, cột thành hàng — công thức `T[j][i] = A[i][j]`.
2. Nắm kích thước: ma trận `m × n` chuyển vị thành `n × m` (ma trận **không vuông** là bẫy chính).
3. Viết 4 cách: hai vòng lặp tạo ma trận mới → list comprehension → `zip(*matrix)` → hoán đổi tại chỗ (chỉ cho ma trận vuông).
4. Hiểu cú pháp `zip`, toán tử trải `*`, và bẫy `[[0] * m] * n`.
5. Trace, chứng minh, test ma trận 1×n, m×1, vuông/không vuông và luyện biến thể (xoay ma trận, đối xứng).

## 1. Đề bài và ví dụ

Cho ma trận số nguyên 2 chiều `matrix`, trả về **ma trận chuyển vị** của nó. Chuyển vị là "lật" ma trận qua **đường chéo chính**, đổi chỉ số hàng và cột cho nhau.

**Ví dụ 1** (3 × 3):
```text
1 2 3        1 4 7
4 5 6   ->   2 5 8
7 8 9        3 6 9
```

**Ví dụ 2** (2 × 3 → 3 × 2):
```text
1 2 3        1 4
4 5 6   ->   2 5
             3 6
```

**Constraints:** `m == len(matrix)`, `n == len(matrix[i])`; `1 <= m, n <= 1000`; `1 <= m·n <= 10^5`; giá trị trong `[-10^9, 10^9]`.

> Quan sát: **hàng `i` của input** trở thành **cột `i` của output**; phần tử `matrix[i][j]` đi tới vị trí `[j][i]`.

In [1]:
matrix = [[1, 2, 3], [4, 5, 6]]
m, n = len(matrix), len(matrix[0])
print("kích thước input :", m, "x", n)
print("kích thước output:", n, "x", m)
for i in range(m):
    for j in range(n):
        print(f"matrix[{i}][{j}] = {matrix[i][j]}  ->  T[{j}][{i}]")

kích thước input : 2 x 3
kích thước output: 3 x 2
matrix[0][0] = 1  ->  T[0][0]
matrix[0][1] = 2  ->  T[1][0]
matrix[0][2] = 3  ->  T[2][0]
matrix[1][0] = 4  ->  T[0][1]
matrix[1][1] = 5  ->  T[1][1]
matrix[1][2] = 6  ->  T[2][1]


## 2. Nền tảng Python cần dùng

### 2.1. Ma trận bằng list lồng nhau, số hàng / số cột

`matrix[i]` là hàng `i`; `matrix[i][j]` là ô hàng `i`, cột `j`. Số hàng `len(matrix)`, số cột `len(matrix[0])` (đề bảo đảm các hàng cùng độ dài).

In [2]:
matrix = [[1, 2, 3], [4, 5, 6]]
print(matrix[1], matrix[1][2], len(matrix), len(matrix[0]))

[4, 5, 6] 6 2 3


### 2.2. Lấy một **cột**

Không có cú pháp trực tiếp để lấy cột của list lồng nhau; phải lấy phần tử `j` của từng hàng: `[row[j] for row in matrix]`. Chuyển vị chính là "lấy lần lượt từng cột làm hàng".

In [3]:
matrix = [[1, 2, 3], [4, 5, 6]]
for j in range(3):
    print("cột", j, "=", [row[j] for row in matrix])

cột 0 = [1, 4]
cột 1 = [2, 5]
cột 2 = [3, 6]


### 2.3. Tạo ma trận rỗng đúng cách

`[[0] * m for _ in range(n)]` tạo `n` hàng **độc lập**, mỗi hàng `m` số 0. Đừng dùng `[[0] * m] * n` — các hàng là **cùng một list** (aliasing).

In [4]:
good = [[0] * 2 for _ in range(3)]
bad = [[0] * 2] * 3
good[0][0] = 1; bad[0][0] = 1
print("good:", good)
print("bad :", bad, "<- cả 3 hàng đều đổi!")
print("bad[0] is bad[1]:", bad[0] is bad[1])

good: [[1, 0], [0, 0], [0, 0]]
bad : [[1, 0], [1, 0], [1, 0]] <- cả 3 hàng đều đổi!
bad[0] is bad[1]: True


### 2.4. `zip` và toán tử trải `*`

- `zip(a, b, c)` ghép phần tử **cùng vị trí** của nhiều dãy thành tuple: phần tử 0 của tất cả, rồi phần tử 1 của tất cả, ...
- `f(*lst)` **trải** list thành các đối số riêng: `zip(*[[1,2,3],[4,5,6]])` = `zip([1,2,3], [4,5,6])`.

Ghép lại: `zip(*matrix)` cho ra **các cột** của ma trận — chính là chuyển vị (dạng tuple).

In [5]:
print(list(zip([1, 2, 3], [4, 5, 6])))
matrix = [[1, 2, 3], [4, 5, 6]]
print(list(zip(*matrix)))
print([list(col) for col in zip(*matrix)])

[(1, 4), (2, 5), (3, 6)]
[(1, 4), (2, 5), (3, 6)]
[[1, 4], [2, 5], [3, 6]]


## 3. Cách 1 — Hai vòng lặp, điền ma trận mới

**Đoạn 1:** `m, n = len(matrix), len(matrix[0])`.

**Đoạn 2:** tạo `result` kích thước **`n × m`** (đảo kích thước!).

**Đoạn 3:** với mọi `i, j`: `result[j][i] = matrix[i][j]`.

In [6]:
def transpose_loops(matrix: list[list[int]]) -> list[list[int]]:
    m, n = len(matrix), len(matrix[0])
    result = [[0] * m for _ in range(n)]      # n hàng, m cột
    for i in range(m):
        for j in range(n):
            result[j][i] = matrix[i][j]
    return result

print(transpose_loops([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))
print(transpose_loops([[1, 2, 3], [4, 5, 6]]))

[[1, 4, 7], [2, 5, 8], [3, 6, 9]]
[[1, 4], [2, 5], [3, 6]]


Mỗi ô được đọc và ghi đúng một lần ⇒ `O(m·n)` thời gian — đây đã là **tối ưu**, vì output có `m·n` ô. Bài này không có "cách nhanh hơn"; các cách sau chỉ khác nhau ở **cách viết** và **bộ nhớ**.

## 4. Insight — Chuyển vị = đọc theo cột, ghi theo hàng

Có hai góc nhìn tương đương:
1. **Theo ô:** `T[j][i] = A[i][j]` — "phân tán" từng ô của `A` tới vị trí mới (cách 1).
2. **Theo hàng của kết quả:** hàng `j` của `T` chính là **cột `j`** của `A` — "gom" từng cột (cách 2 và 3).

Góc nhìn 2 cho phép viết gọn bằng list comprehension: `T = [[A[i][j] for i in range(m)] for j in range(n)]`.

**Ma trận vuông** (`m == n`) còn cho phép làm **tại chỗ**: hoán đổi `A[i][j]` với `A[j][i]` cho các ô **phía trên đường chéo** (`j > i`). Ma trận không vuông thì không làm tại chỗ được vì kích thước thay đổi.

In [7]:
A = [[1, 2, 3], [4, 5, 6]]
m, n = len(A), len(A[0])
for j in range(n):
    print(f"hàng {j} của T = cột {j} của A =", [A[i][j] for i in range(m)])

hàng 0 của T = cột 0 của A = [1, 4]
hàng 1 của T = cột 1 của A = [2, 5]
hàng 2 của T = cột 2 của A = [3, 6]


## 5. Cách 2 — List comprehension lồng nhau

Đọc từ ngoài vào trong: với mỗi `j` (hàng của kết quả), tạo list `[matrix[i][j] for i in range(m)]` (cột `j` của input).

In [8]:
def transpose_comprehension(matrix: list[list[int]]) -> list[list[int]]:
    m, n = len(matrix), len(matrix[0])
    return [[matrix[i][j] for i in range(m)] for j in range(n)]

print(transpose_comprehension([[1, 2, 3], [4, 5, 6]]))

[[1, 4], [2, 5], [3, 6]]


## 6. Cách 3 — `zip(*matrix)` (thành ngữ Python)

Một dòng, rất hay dùng trong thực tế. Nhớ đổi tuple thành list nếu đề yêu cầu `list[list[int]]`.

In [9]:
def transpose_zip(matrix: list[list[int]]) -> list[list[int]]:
    return [list(col) for col in zip(*matrix)]

print(transpose_zip([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))
print(transpose_zip([[1, 2, 3], [4, 5, 6]]))

[[1, 4, 7], [2, 5, 8], [3, 6, 9]]
[[1, 4], [2, 5], [3, 6]]


### Vì sao `zip(*matrix)` đúng?

`zip(*matrix)` = `zip(matrix[0], matrix[1], ..., matrix[m-1])`. Tuple thứ `j` mà `zip` sinh ra là `(matrix[0][j], matrix[1][j], ..., matrix[m-1][j])` — đúng cột `j`, tức hàng `j` của chuyển vị.

## 7. TRACE từng bước

### 7.1. Bảng trace cách 1 cho `[[1, 2, 3], [4, 5, 6]]` (m = 2, n = 3)

`result` khởi tạo 3 hàng × 2 cột toàn 0.

| bước | i | j | matrix[i][j] | ghi vào | result sau bước |
|---|---|---|---|---|---|
| 1 | 0 | 0 | 1 | result[0][0] | [[1,0],[0,0],[0,0]] |
| 2 | 0 | 1 | 2 | result[1][0] | [[1,0],[2,0],[0,0]] |
| 3 | 0 | 2 | 3 | result[2][0] | [[1,0],[2,0],[3,0]] |
| 4 | 1 | 0 | 4 | result[0][1] | [[1,4],[2,0],[3,0]] |
| 5 | 1 | 1 | 5 | result[1][1] | [[1,4],[2,5],[3,0]] |
| 6 | 1 | 2 | 6 | result[2][1] | [[1,4],[2,5],[3,6]] |

Hàng 0 của input (`1 2 3`) được rải xuống **cột 0** của kết quả.

In [10]:
def trace_transpose(matrix):
    m, n = len(matrix), len(matrix[0])
    result = [[0] * m for _ in range(n)]
    print(f"input {m}x{n} -> output {n}x{m}; result ban đầu = {result}")
    step = 1
    for i in range(m):
        for j in range(n):
            result[j][i] = matrix[i][j]
            print(f"bước {step}: i={i} j={j} matrix[i][j]={matrix[i][j]:>2} -> result[{j}][{i}]  result={result}")
            step += 1
    return result

trace_transpose([[1, 2, 3], [4, 5, 6]]);

input 2x3 -> output 3x2; result ban đầu = [[0, 0], [0, 0], [0, 0]]
bước 1: i=0 j=0 matrix[i][j]= 1 -> result[0][0]  result=[[1, 0], [0, 0], [0, 0]]
bước 2: i=0 j=1 matrix[i][j]= 2 -> result[1][0]  result=[[1, 0], [2, 0], [0, 0]]
bước 3: i=0 j=2 matrix[i][j]= 3 -> result[2][0]  result=[[1, 0], [2, 0], [3, 0]]
bước 4: i=1 j=0 matrix[i][j]= 4 -> result[0][1]  result=[[1, 4], [2, 0], [3, 0]]
bước 5: i=1 j=1 matrix[i][j]= 5 -> result[1][1]  result=[[1, 4], [2, 5], [3, 0]]
bước 6: i=1 j=2 matrix[i][j]= 6 -> result[2][1]  result=[[1, 4], [2, 5], [3, 6]]


### 7.2. Trace `zip(*matrix)` — xem từng tuple

In [11]:
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
for j, col in enumerate(zip(*matrix)):
    print(f"tuple thứ {j} từ zip = {col}  (= cột {j})")

tuple thứ 0 từ zip = (1, 4, 7)  (= cột 0)
tuple thứ 1 từ zip = (2, 5, 8)  (= cột 1)
tuple thứ 2 từ zip = (3, 6, 9)  (= cột 2)


## 8. Code nộp LeetCode

In [12]:
class Solution:
    def transpose(self, matrix: list[list[int]]) -> list[list[int]]:
        m, n = len(matrix), len(matrix[0])
        result = [[0] * m for _ in range(n)]
        for i in range(m):
            for j in range(n):
                result[j][i] = matrix[i][j]
        return result

sol = Solution()
assert sol.transpose([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
assert sol.transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]
print("Solution PASS 2 ví dụ đề")

Solution PASS 2 ví dụ đề


Bản một dòng (cũng được chấp nhận):

In [13]:
class SolutionZip:
    def transpose(self, matrix: list[list[int]]) -> list[list[int]]:
        return [list(col) for col in zip(*matrix)]

print(SolutionZip().transpose([[1, 2, 3], [4, 5, 6]]))

[[1, 4], [2, 5], [3, 6]]


## 9. Vì sao đúng? Độ phức tạp

**Đúng (cách 1):** `result` có `n` hàng, `m` cột — đúng kích thước chuyển vị. Hai vòng lặp đi qua **mọi** cặp `(i, j)` với `0 <= i < m`, `0 <= j < n`, và mỗi cặp ghi đúng một ô `result[j][i]`. Ánh xạ `(i, j) → (j, i)` là **song ánh** giữa các ô của input và output, nên mọi ô output được ghi đúng một lần với đúng giá trị `matrix[i][j]`.

**Tính chất kiểm chứng:** chuyển vị hai lần ra lại ma trận gốc: `(Aᵀ)ᵀ = A`.

**Độ phức tạp:** ma trận `m × n`.

| Cách | Time | Space phụ (ngoài output) | Ghi chú |
|---|---|---|---|
| 1. Hai vòng lặp | `O(m·n)` | `O(1)` | Rõ ràng, dễ giải thích |
| 2. Comprehension | `O(m·n)` | `O(1)` | Gọn |
| 3. `zip(*matrix)` | `O(m·n)` | `O(1)` (tuple tạm từng cột) | Thành ngữ Python |
| 4. Tại chỗ (vuông) | `O(n²)` | `O(1)` — **không** tạo output mới | Chỉ khi `m == n` |

## 10. Cách 4 — Chuyển vị **tại chỗ** cho ma trận vuông

Với `n × n`, chỉ cần hoán đổi các cặp đối xứng qua đường chéo: duyệt `i` từ `0` tới `n-1`, `j` từ `i + 1` tới `n-1`. Không duyệt `j <= i`, vì nếu duyệt cả hai nửa thì mỗi cặp bị hoán đổi **hai lần** → về lại như cũ!

In [14]:
def transpose_inplace_square(a: list[list[int]]) -> list[list[int]]:
    n = len(a)
    for i in range(n):
        for j in range(i + 1, n):
            a[i][j], a[j][i] = a[j][i], a[i][j]
    return a

sq = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print(transpose_inplace_square(sq))

[[1, 4, 7], [2, 5, 8], [3, 6, 9]]


In [15]:
def trace_inplace(a):
    n = len(a)
    for i in range(n):
        for j in range(i + 1, n):
            a[i][j], a[j][i] = a[j][i], a[i][j]
            print(f"đổi ({i},{j}) <-> ({j},{i}): {a}")
    return a

trace_inplace([[1, 2, 3], [4, 5, 6], [7, 8, 9]]);

đổi (0,1) <-> (1,0): [[1, 4, 3], [2, 5, 6], [7, 8, 9]]
đổi (0,2) <-> (2,0): [[1, 4, 7], [2, 5, 6], [3, 8, 9]]
đổi (1,2) <-> (2,1): [[1, 4, 7], [2, 5, 8], [3, 6, 9]]


## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Hai ví dụ đề.
2. `1 × 1`; một hàng `1 × n` (thành một cột); một cột `m × 1` (thành một hàng).
3. Số âm, số lớn.
4. Tính chất: chuyển vị hai lần ra gốc; kích thước đổi `m × n → n × m`.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.transpose([[1, 2, 3], [4, 5, 6], [7, 8, 9]]), [[1, 4, 7], [2, 5, 8], [3, 6, 9]])
check("đề 2", sol.transpose([[1, 2, 3], [4, 5, 6]]), [[1, 4], [2, 5], [3, 6]])
check("1x1", sol.transpose([[7]]), [[7]])
check("1 hàng", sol.transpose([[1, 2, 3, 4]]), [[1], [2], [3], [4]])
check("1 cột", sol.transpose([[1], [2], [3]]), [[1, 2, 3]])
check("số âm/lớn", sol.transpose([[-10**9, 5], [0, 10**9]]), [[-10**9, 0], [5, 10**9]])

PASS đề 1: [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
PASS đề 2: [[1, 4], [2, 5], [3, 6]]
PASS 1x1: [[7]]
PASS 1 hàng: [[1], [2], [3], [4]]
PASS 1 cột: [[1, 2, 3]]
PASS số âm/lớn: [[-1000000000, 0], [5, 1000000000]]


### 11.1. Đối chiếu các cài đặt + tính chất (có seed)

In [17]:
import random, copy
random.seed(867)
impls = [transpose_loops, transpose_comprehension, transpose_zip, Solution().transpose, SolutionZip().transpose]
for _ in range(300):
    m, n = random.randint(1, 6), random.randint(1, 6)
    A = [[random.randint(-9, 9) for _ in range(n)] for _ in range(m)]
    expected = transpose_loops(A)
    for f in impls:
        assert f(A) == expected, (A, f)
    assert len(expected) == n and all(len(r) == m for r in expected)   # kích thước n x m
    assert transpose_loops(expected) == A                              # (A^T)^T = A
    if m == n:
        assert transpose_inplace_square(copy.deepcopy(A)) == expected
print("300 ma trận ngẫu nhiên: mọi cài đặt khớp; kích thước và (A^T)^T = A đúng")

300 ma trận ngẫu nhiên: mọi cài đặt khớp; kích thước và (A^T)^T = A đúng


### 11.2. Hiệu năng với `m·n = 10^5`

In [18]:
import time
big = [[random.randint(-10**9, 10**9) for _ in range(1000)] for _ in range(100)]
for name, f in [("hai vòng lặp", transpose_loops), ("zip", transpose_zip)]:
    t0 = time.perf_counter(); r = f(big); t1 = time.perf_counter()
    print(f"{name:13}: {t1 - t0:.4f}s, kích thước {len(r)}x{len(r[0])}")

hai vòng lặp : 0.0037s, kích thước 1000x100
zip          : 0.0031s, kích thước 1000x100


## 12. Lỗi người mới hay gặp

### Lỗi 1: tạo `result` sai kích thước `m × n` (chỉ đúng khi vuông)

In [19]:
def wrong_shape(matrix):
    m, n = len(matrix), len(matrix[0])
    result = [[0] * n for _ in range(m)]            # SAI: m hàng, n cột
    for i in range(m):
        for j in range(n):
            result[j][i] = matrix[i][j]
    return result

try:
    print(wrong_shape([[1, 2, 3], [4, 5, 6]]))
except IndexError as e:
    print("IndexError:", e, "-> phải tạo n hàng, m cột")
print("vuông thì 'tình cờ' đúng:", wrong_shape([[1, 2], [3, 4]]))

IndexError: list index out of range -> phải tạo n hàng, m cột
vuông thì 'tình cờ' đúng: [[1, 3], [2, 4]]


### Lỗi 2: `[[0] * m] * n` — mọi hàng cùng một list

In [20]:
def wrong_alias(matrix):
    m, n = len(matrix), len(matrix[0])
    result = [[0] * m] * n                          # SAI: aliasing
    for i in range(m):
        for j in range(n):
            result[j][i] = matrix[i][j]
    return result

print("sai :", wrong_alias([[1, 2, 3], [4, 5, 6]]))
print("đúng:", Solution().transpose([[1, 2, 3], [4, 5, 6]]))

sai : [[3, 6], [3, 6], [3, 6]]
đúng: [[1, 4], [2, 5], [3, 6]]


### Lỗi 3: hoán đổi tại chỗ trên **cả** ma trận (`j` chạy từ 0)

Mỗi cặp bị đổi hai lần → không đổi gì.

In [21]:
def wrong_double_swap(a):
    n = len(a)
    for i in range(n):
        for j in range(n):                           # SAI: phải là range(i + 1, n)
            a[i][j], a[j][i] = a[j][i], a[i][j]
    return a

print("sai :", wrong_double_swap([[1, 2], [3, 4]]))
print("đúng:", [[1, 3], [2, 4]])

sai : [[1, 2], [3, 4]]
đúng: [[1, 3], [2, 4]]


### Lỗi 4: trả về `list(zip(*matrix))` — list các **tuple**

LeetCode thường vẫn chấp nhận khi so sánh, nhưng kiểu dữ liệu sai so với chữ ký `list[list[int]]`, và `assert` trong Python sẽ thấy `(1, 4) != [1, 4]`.

In [22]:
res = list(zip(*[[1, 2], [3, 4]]))
print(res, "| bằng [[1, 3], [2, 4]]?", res == [[1, 3], [2, 4]])
print([list(c) for c in res] == [[1, 3], [2, 4]])

[(1, 3), (2, 4)] | bằng [[1, 3], [2, 4]]? False
True


## 13. Mẹo nhận dạng pattern

- "Đổi hàng thành cột", "lật qua đường chéo", "đọc theo cột" → chuyển vị: `T[j][i] = A[i][j]` hoặc `zip(*A)`.
- Bài ma trận: **luôn** ghi rõ `m` (số hàng), `n` (số cột), kiểm tra lại kích thước output trước khi code.
- Nhiều phép biến đổi ma trận = **tổ hợp** các phép đơn giản: xoay 90° = chuyển vị + lật ngang; xoay −90° = chuyển vị + lật dọc.
- Bài liên quan: *Rotate Image (48)*, *Flipping an Image (832)*, *Reshape the Matrix (566)*, *Toeplitz Matrix (766)*, *Spiral Matrix (54)*.

## 14. Biến thể

### 14.1. Xoay 90° chiều kim đồng hồ = chuyển vị rồi đảo mỗi hàng

In [23]:
def rotate_cw(a):
    return [list(col)[::-1] for col in zip(*a)]

def rotate_ccw(a):
    return [list(col) for col in zip(*a)][::-1]

A = [[1, 2, 3], [4, 5, 6]]
print("cw :", rotate_cw(A))
print("ccw:", rotate_ccw(A))

cw : [[4, 1], [5, 2], [6, 3]]
ccw: [[3, 6], [2, 5], [1, 4]]


### 14.2. Ma trận đối xứng: `A == Aᵀ`

In [24]:
def is_symmetric(a):
    return a == [list(c) for c in zip(*a)]

print(is_symmetric([[1, 2], [2, 5]]), is_symmetric([[1, 2], [3, 4]]))

True False


### 14.3. Chuyển vị theo **đường chéo phụ** (anti-transpose)

`T[n-1-j][m-1-i] = A[i][j]` — tương đương xoay 180° rồi chuyển vị.

In [25]:
def anti_transpose(a):
    m, n = len(a), len(a[0])
    res = [[0] * m for _ in range(n)]
    for i in range(m):
        for j in range(n):
            res[n - 1 - j][m - 1 - i] = a[i][j]
    return res

print(anti_transpose([[1, 2, 3], [4, 5, 6]]))

[[6, 3], [5, 2], [4, 1]]


## 15. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
Chuyển vị `[[1, 2], [3, 4], [5, 6], [7, 8]]` (4 × 2). Kết quả kích thước bao nhiêu?

### Bài 2 — Dự đoán output
Không chạy, đoán: `list(zip("ab", "cd"))`, `list(zip(*["ab", "cd"]))`, `print(*[1, 2, 3])`.

### Bài 3 — Điền TODO cho cách hai vòng lặp

In [26]:
def ex3_transpose(matrix: list[list[int]]) -> list[list[int]]:
    m, n = len(matrix), len(matrix[0])
    # TODO: tạo result n hàng, m cột
    # TODO: hai vòng lặp, result[j][i] = matrix[i][j]
    pass

# assert ex3_transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]

### Bài 4 — Reshape the Matrix (LeetCode 566)
Đổi ma trận `m × n` thành `r × c` (đọc theo hàng); nếu `m·n != r·c` thì trả ma trận gốc.

In [27]:
def ex4_reshape(mat: list[list[int]], r: int, c: int) -> list[list[int]]:
    # TODO
    pass

# assert ex4_reshape([[1, 2], [3, 4]], 1, 4) == [[1, 2, 3, 4]]
# assert ex4_reshape([[1, 2], [3, 4]], 2, 4) == [[1, 2], [3, 4]]

### Bài 5 — Toeplitz Matrix (LeetCode 766)
Ma trận Toeplitz nếu mọi đường chéo từ trên-trái xuống dưới-phải có cùng giá trị. Kiểm tra `matrix[i][j] == matrix[i-1][j-1]`.

In [28]:
def ex5_is_toeplitz(matrix: list[list[int]]) -> bool:
    # TODO
    pass

# assert ex5_is_toeplitz([[1, 2, 3, 4], [5, 1, 2, 3], [9, 5, 1, 2]]) is True
# assert ex5_is_toeplitz([[1, 2], [2, 2]]) is False

### Bài 6 — Mini challenge: xoay 90° **tại chỗ** (LeetCode 48)
Dùng chuyển vị tại chỗ (cách 4) rồi đảo từng hàng.

In [29]:
def ex6_rotate_inplace(a: list[list[int]]) -> None:
    # TODO
    pass

# M = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; ex6_rotate_inplace(M); assert M == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]

## 16. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 1: cột 0 = `[1, 3, 5, 7]` thành hàng 0.
- Bài 4: "duỗi" ma trận thành một dãy rồi cắt lại theo `c` phần tử.
- Bài 6: xoay = chuyển vị + lật ngang.

**Mức 2 — cú pháp**
- Bài 3: `[[0] * m for _ in range(n)]`.
- Bài 4: `flat = [x for row in mat for x in row]` (comprehension hai tầng — đọc như hai vòng `for` lồng nhau).
- Bài 5: duyệt `i` từ 1, `j` từ 1.

**Mức 3 — gần code**
- Bài 4: `return [flat[k * c:(k + 1) * c] for k in range(r)]`.
- Bài 6: `for i in range(n): for j in range(i + 1, n): swap`; rồi `for row in a: row.reverse()`.

## 17. Đáp án tham khảo

**Bài 1:** `[[1, 3, 5, 7], [2, 4, 6, 8]]`, kích thước 2 × 4.

**Bài 2:** `[('a', 'c'), ('b', 'd')]`; `[('a', 'c'), ('b', 'd')]` (chuỗi cũng là dãy); `1 2 3`.

In [30]:
print(list(zip("ab", "cd")), list(zip(*["ab", "cd"])))
print(*[1, 2, 3])
assert Solution().transpose([[1, 2], [3, 4], [5, 6], [7, 8]]) == [[1, 3, 5, 7], [2, 4, 6, 8]]

def ans3_transpose(matrix):
    m, n = len(matrix), len(matrix[0])
    result = [[0] * m for _ in range(n)]
    for i in range(m):
        for j in range(n):
            result[j][i] = matrix[i][j]
    return result

def ans4_reshape(mat, r, c):
    m, n = len(mat), len(mat[0])
    if m * n != r * c:
        return mat
    flat = [x for row in mat for x in row]
    return [flat[k * c:(k + 1) * c] for k in range(r)]

def ans5_is_toeplitz(matrix):
    for i in range(1, len(matrix)):
        for j in range(1, len(matrix[0])):
            if matrix[i][j] != matrix[i - 1][j - 1]:
                return False
    return True

def ans6_rotate_inplace(a):
    n = len(a)
    for i in range(n):
        for j in range(i + 1, n):
            a[i][j], a[j][i] = a[j][i], a[i][j]
    for row in a:
        row.reverse()

assert ans3_transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]
assert ans4_reshape([[1, 2], [3, 4]], 1, 4) == [[1, 2, 3, 4]]
assert ans4_reshape([[1, 2], [3, 4]], 2, 4) == [[1, 2], [3, 4]]
assert ans5_is_toeplitz([[1, 2, 3, 4], [5, 1, 2, 3], [9, 5, 1, 2]]) is True
assert ans5_is_toeplitz([[1, 2], [2, 2]]) is False
M = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; ans6_rotate_inplace(M); assert M == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
assert M == rotate_cw([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("Đáp án bài 1-6 PASS")

[('a', 'c'), ('b', 'd')] [('a', 'c'), ('b', 'd')]
1 2 3
Đáp án bài 1-6 PASS


## 18. Tổng kết — template nhớ nhanh và checklist submit

```python
m, n = len(matrix), len(matrix[0])
result = [[0] * m for _ in range(n)]      # n hàng, m cột!
for i in range(m):
    for j in range(n):
        result[j][i] = matrix[i][j]
return result
# hoặc: return [list(col) for col in zip(*matrix)]
```

**Câu thần chú:** *"`m × n` thành `n × m`; ô `[i][j]` đi tới `[j][i]`; `zip(*A)` cho các cột."*

**Checklist trước khi submit:**
- [ ] Output có **n hàng, m cột** (không phải m × n)?
- [ ] Tạo ma trận bằng comprehension, **không** dùng `[[0] * m] * n`?
- [ ] Nếu dùng `zip`, đã đổi tuple thành list?
- [ ] Tại chỗ chỉ khi ma trận **vuông**, và `j` chạy từ `i + 1`?
- [ ] Đã test `1 × n`, `m × 1`, `1 × 1`?